### dependencies & globals


In [1]:
import requests
import pandas as pd
import geopandas as gpd

### API call to get GDOT projects


In [ ]:
# Define the base URL of the MapServer layer
base_url = "https://rnhp.dot.ga.gov/hosting/rest/services/GDOT_Public_Outreach/Hub_Project_Search/MapServer/2/query"

# Define query parameters
params = {
    # filter by construction status to include both under construction and pre-construction
    "where": "CONSTRUTION_STATUS_DERIVED = 'PRE-CONSTRUCTION' OR CONSTRUTION_STATUS_DERIVED = 'UNDER CONSTRUCTION'",
    "outFields": "*",  # Get all fields
    "f": "geojson",  # Request data in GeoJSON format
    "returnGeometry": "true",  # Include geometry data
    "resultOffset": 0,  # Start from the first record
    "resultRecordCount": 1000  # Get 1000 records
}

# list to store each batch of GeoDataFrames
gdfs = []

while True:

    # Make the GET request
    response = requests.get(base_url, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        batch_gdf = gpd.GeoDataFrame.from_features(data["features"])

        # if no more data is returned, break the loop
        if batch_gdf.empty:
            break

        gdfs.append(batch_gdf)

        print(
            f"✅ Fetched {len(batch_gdf)} records (offset: {params['resultOffset']})")

        # increase offset by the batch size to fetch the next chunk
        params["resultOffset"] += params["resultRecordCount"]

    else:
        print(
            f"❌ Error fetching data: {response.status_code} - {response.text}")
        break

# Filter out empty GeoDataFrames
gdfs = [gdf.dropna(axis=1, how='all') for gdf in gdfs if not gdf.empty]

# Combine all batches into a single GeoDataFrame
if gdfs:
    full_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

else:
    print("❌ No data retrieved.")

# drop any columns that ONLY contain NaN values
full_gdf = full_gdf.dropna(axis=1, how="all")

# replace all-caps values in the 'CONSTRUTION_STATUS_DERIVED' column
full_gdf['CONSTRUTION_STATUS_DERIVED'] = full_gdf['CONSTRUTION_STATUS_DERIVED'].str.replace(
    'PRE-CONSTRUCTION', 'Pre-Construction')
full_gdf['CONSTRUTION_STATUS_DERIVED'] = full_gdf['CONSTRUTION_STATUS_DERIVED'].str.replace(
    'UNDER CONSTRUCTION', 'Under Construction')

# define a list of sub-strings to keep in all caps; everything else in the 'SHORT_DESCR' column should be converted to title case
keep_all_caps = ["SR ", "CO ", "CR ", "US ", "RD", "SO ",
                 "CS ", "MI ", "SE ", "NE ", "SW ", "NW ", "NS ", "CSX", "CCTV", "-TIA", " - TIA"]
full_gdf['SHORT_DESCR'] = full_gdf['SHORT_DESCR'].str.title()
for sub_str in keep_all_caps:
    full_gdf['SHORT_DESCR'] = full_gdf['SHORT_DESCR'].str.replace(
        sub_str.title(), sub_str)

# convert Unix timestamp to date
full_gdf['LAST_REFRESH_DTTM'] = pd.to_datetime(
    full_gdf['LAST_REFRESH_DTTM'], unit='ms').dt.date
full_gdf['CURR_COMPLETION_DATE'] = pd.to_datetime(
    full_gdf['CURR_COMPLETION_DATE'], unit='ms').dt.date
full_gdf['AWARD_DATE'] = pd.to_datetime(
    full_gdf['AWARD_DATE'], unit='ms').dt.date
full_gdf['TPRO_PROJ_COMPLETE_DT'] = pd.to_datetime(
    full_gdf['TPRO_PROJ_COMPLETE_DT'], unit='ms').dt.date

# filter out records that have either 'WPH' (completed) or 'REJ' (rejected) in the 'CONST_STAT_CD' column
full_gdf = full_gdf[~full_gdf['CONST_STAT_CD'].isin(['WPH', 'REJ'])]

# rename columns
full_gdf = full_gdf.rename(columns={
    'LAST_REFRESH_DTTM': 'Last_refresh',
    'CURR_COMPLETION_DATE': 'Completion_date',
    'AWARD_DATE': 'Award_date',
    'PROJECT_COUNTIES': 'Project_counties',
    'PROJ_ID': 'Project_ID',
    'SHORT_DESCR': 'Description',
    'CONTRACT_DESCRIPTION': 'UC_description',
    'CONTRACTOR_NAME': 'Contractor_name',
    'IS_TIA_PROJECT': 'Is_TIA_project',
    'CONSTRUCTION_PERCENT_COMPLETE': 'Construction_percent_complete',
    'CONSTRUTION_STATUS_DERIVED': 'Construction_status',
})

# drop unneeded columns
full_gdf = full_gdf.drop(columns=[
    'OBJECTID',
    'PRIORITY_CD',
    'SOURCE_OF_CONSTRUCTION_DATES',
    'CONTRACT_ID',
    'CONSTRUTION_STATUS_DERIVED_RSN',
    'PAYMENT_PERCENT_COMPLETE',
    'ESRI_OID',
    'REC_STATUS',
    'LET_RESP_CD',
    'PRIORITY_CD_DESCR',
    'CONST_STAT_CD',
    'TPRO_PROJ_COMPLETE_DT'
])

# create URL column
full_gdf['Project_URL'] = 'https://www.dot.ga.gov/applications/geopi/Pages/Dashboard.aspx?ProjectId=' + \
    full_gdf['Project_ID'].astype(str)

# export full_gdf to a geojson file inside the 'data' folder
full_gdf.to_file("data/GDOT_export.geojson", driver="GeoJSON")

# print status
print(f"✅ Successfully fetched {len(full_gdf):,} records")

# Display final GeoDataFrame
full_gdf

✅ Fetched 1000 records (offset: 0)
✅ Fetched 1000 records (offset: 1000)
✅ Fetched 1000 records (offset: 2000)
✅ Fetched 529 records (offset: 3000)


/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.11/site-packages/pyogrio/geopandas.py:662: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


✅ Successfully fetched 3,408 records


,geometry,Project_ID,Description,Project_counties,Is_TIA_project,Contractor_name,UC_description,Award_date,Completion_date,Construction_percent_complete,Construction_status,Last_refresh,Project_URL
0,"LINESTRING (-84.7983 33.57898, -84.79799 33.57...",0017780,CS 435/Garretts Ferry Road @ Chattahoochee Riv...,Fulton,No,C & S CONTRACTING CO LLC,GARRETTS FERRY RD (CR 435) - BRDG REHAB,2023-06-30,2024-09-13,73.72,Under Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
1,"LINESTRING (-84.37062 34.9651, -84.37027 34.96...",0017782,SR 5 From Old Flowers Road To SR 60; Inc Round...,Fannin,No,C W MATTHEWS CONTRACTING,SR 5 (BLUE RIDGE DR) - TRAFFIC MANAGEMENT,2023-05-05,2026-03-31,11.79,Under Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
2,"LINESTRING (-84.4939 34.24325, -84.49388 34.24...",0017789,SR 140 From SR 5 Bu To N Of CS 662/Mary Lane I...,Cherokee,No,None,None,NaT,NaT,NaN,Pre-Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
3,"LINESTRING (-84.48954 33.69139, -84.48926 33.6...",0017802,Atlanta Traffic Signal Enhancement Program - P...,Fulton,No,None,None,NaT,NaT,NaN,Pre-Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
4,"LINESTRING (-84.45336 34.20175, -84.45375 34.2...",0017804,SR 140 @ CR 309/Univeter Road,Cherokee,No,None,None,NaT,NaT,NaN,Pre-Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3524,"LINESTRING (-84.81271 33.2236, -84.81165 33.22...",M006068,I-85 From Meriwether County Line To SR 74,"Coweta , Fulton",No,PEEK PAVEMENT MARKING LLC,I-85/SR 403 - PVMNT MRKG,2023-09-01,2024-12-10,85.18,Under Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
3525,"LINESTRING (-85.5537 34.71385, -85.55158 34.71...",M006071,I-59 From Alabama State Line To SR 136,Dade,No,None,None,NaT,NaT,NaN,Pre-Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
3526,"LINESTRING (-84.95134 32.51723, -84.95171 32.5...",M006072,I-185 From S Of CR 1379/Armour Road/Muscogee T...,"Harris , Muscogee , Troup",No,None,None,NaT,NaT,NaN,Pre-Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...
3527,"LINESTRING (-84.44259 33.90154, -84.44147 33.9...",M006073,I-285 From Chattahoochee River To W Of SR 400,Fulton,No,None,None,NaT,NaT,NaN,Pre-Construction,2025-02-19,https://www.dot.ga.gov/applications/geopi/Page...


### misc cleaning


In [ ]:
full_gdf.drop(columns=['geometry']).to_csv(
    'GDOT_export_noGeo.csv', index=False)